Setup & Model Loading

In [1]:
# FASE 9 — Cell 1: Setup & Load Semua Artifacts
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json
from pathlib import Path

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score,
    recall_score, mean_absolute_error,
    mean_squared_error, r2_score,
    ConfusionMatrixDisplay, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize
import tensorflow as tf
from tensorflow.keras.models import load_model

warnings.filterwarnings("ignore")

# ─── PATH SETUP ────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import (
    DATA_PROCESSED_DIR, MODELS_ML_DIR, MODELS_DL_DIR,
    MODELS_FINAL_DIR, GLOBAL_SEED, LABEL_MAP,
)

np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

SEP = "=" * 65
sep = "-" * 65

print(SEP)
print("  FASE 9 — EVALUATION, CALIBRATION & MODEL SELECTION")
print(SEP)

# ─── LOAD CLASSIFIER DATA ──────────────────────────────────
print("\n  [1/4] Memuat data klasifikasi...")
X_val_clf  = pd.read_parquet(DATA_PROCESSED_DIR / "X_val_clf.parquet")
y_val_clf  = pd.read_parquet(DATA_PROCESSED_DIR / "y_val_clf.parquet").squeeze()
X_test_clf = pd.read_parquet(DATA_PROCESSED_DIR / "X_test_clf.parquet")
y_test_clf = pd.read_parquet(DATA_PROCESSED_DIR / "y_test_clf.parquet").squeeze()
print("  OK — Classifier data loaded.")

# ─── LOAD RUL DATA ─────────────────────────────────────────
print("\n  [2/4] Memuat data RUL...")
X_val_rul  = pd.read_parquet(DATA_PROCESSED_DIR / "X_val_rul.parquet")
y_val_rul  = pd.read_parquet(DATA_PROCESSED_DIR / "y_val_rul.parquet").squeeze()
X_test_rul = pd.read_parquet(DATA_PROCESSED_DIR / "X_test_rul.parquet")
y_test_rul = pd.read_parquet(DATA_PROCESSED_DIR / "y_test_rul.parquet").squeeze()

df_ssbs = (
    pd.read_parquet(DATA_PROCESSED_DIR / "df_ssbs_rul.parquet")
    .sort_values(["machine_id", "timestamp"])
    .reset_index(drop=True)
)

VAL_MACHINES  = ["M-15", "M-16", "M-17"]
TEST_MACHINES = ["M-18", "M-19", "M-20"]

mask_val  = (df_ssbs[df_ssbs["machine_id"].isin(VAL_MACHINES)]
             ["health_label_encoded"].isin([1, 2]).values)
mask_test = (df_ssbs[df_ssbs["machine_id"].isin(TEST_MACHINES)]
             ["health_label_encoded"].isin([1, 2]).values)

X_val_rul_f  = X_val_rul[mask_val].reset_index(drop=True)
y_val_rul_f  = y_val_rul[mask_val].reset_index(drop=True)
X_test_rul_f = X_test_rul[mask_test].reset_index(drop=True)
y_test_rul_f = y_test_rul[mask_test].reset_index(drop=True)
print("  OK — RUL data loaded & filtered (WARNING+CRITICAL only).")

# ─── SEQUENCE PREPARATION UNTUK LSTM ──────────────────────
SEQ_LEN = 24

def create_sequences(X, y, seq_len):
    seqs, tgts = [], []
    for i in range(seq_len, len(X)):
        seqs.append(X[i - seq_len:i])
        tgts.append(y[i])
    return np.array(seqs), np.array(tgts)

X_val_seq,  y_val_seq  = create_sequences(
    X_val_rul_f.values,  y_val_rul_f.values,  SEQ_LEN)
X_test_seq, y_test_seq = create_sequences(
    X_test_rul_f.values, y_test_rul_f.values, SEQ_LEN)
print("  OK — Sequences created.")

# ─── LOAD MODELS ───────────────────────────────────────────
print("\n  [3/4] Memuat semua model...")
rf_model   = joblib.load(MODELS_ML_DIR / "rf_classifier.pkl")
xgb_model  = joblib.load(MODELS_ML_DIR / "xgb_classifier.pkl")
lgbm_model = joblib.load(MODELS_ML_DIR / "lgbm_classifier.pkl")
xgb_rul    = joblib.load(MODELS_ML_DIR / "xgb_regressor.pkl")
lstm_model  = load_model(str(MODELS_DL_DIR / "lstm_rul_best_v2.keras"))

XGB_CLF_THRESHOLD = 0.60
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}   # {0: 'HEALTHY', 1: 'WARNING', 2: 'CRITICAL'}
CLASS_NAMES = ["HEALTHY", "WARNING", "CRITICAL"]

print("  OK — Semua model loaded.")

# ─── KONFIRMASI TABEL MODEL ────────────────────────────────
print(f"\n{sep}")
print("  MODEL REGISTRY")
print(sep)
model_registry = [
    ("Random Forest",      "CLF", "rf_classifier.pkl",      rf_model),
    ("XGBoost Clf",        "CLF", "xgb_classifier.pkl",     xgb_model),
    ("LightGBM Clf",       "CLF", "lgbm_classifier.pkl",    lgbm_model),
    ("XGBoost Regressor",  "RUL", "xgb_regressor.pkl",      xgb_rul),
    ("LSTM V2",            "RUL", "lstm_rul_best_v2.keras",  lstm_model),
]
print(f"\n  {'Model':<22} {'Type':<6} {'File':<28} {'Status'}")
print(f"  {'-'*65}")
for name, mtype, fname, _ in model_registry:
    print(f"  {name:<22} {mtype:<6} {fname:<28} ✅")

# ─── KONFIRMASI TABEL DATA ─────────────────────────────────
print(f"\n{sep}")
print("  DATA SHAPES")
print(sep)
datasets = [
    ("X_val_clf",   X_val_clf),
    ("X_test_clf",  X_test_clf),
    ("X_val_rul_f", X_val_rul_f),
    ("X_test_rul_f",X_test_rul_f),
    ("X_val_seq",   X_val_seq),
    ("X_test_seq",  X_test_seq),
]
print(f"\n  {'Dataset':<16} {'Shape'}")
print(f"  {'-'*35}")
for name, arr in datasets:
    shape = arr.shape if hasattr(arr, "shape") else (len(arr),)
    print(f"  {name:<16} {str(shape)}")

print(f"\n  XGB_CLF_THRESHOLD  : {XGB_CLF_THRESHOLD}")
print(f"  GLOBAL_SEED        : {GLOBAL_SEED}")
print(f"  SEQ_LEN (LSTM)     : {SEQ_LEN}")
print(f"  LABEL_MAP          : {LABEL_MAP}")

print(f"\n{SEP}")
print("  ✅ FASE 9 SETUP COMPLETE — Semua artifacts loaded")
print(f"  [4/4] Lanjut ke Cell 2 untuk evaluasi Model 1 (Classifier).")
print(SEP)



  FASE 9 — EVALUATION, CALIBRATION & MODEL SELECTION

  [1/4] Memuat data klasifikasi...
  OK — Classifier data loaded.

  [2/4] Memuat data RUL...
  OK — RUL data loaded & filtered (WARNING+CRITICAL only).
  OK — Sequences created.

  [3/4] Memuat semua model...

  OK — Semua model loaded.

-----------------------------------------------------------------
  MODEL REGISTRY
-----------------------------------------------------------------

  Model                  Type   File                         Status
  -----------------------------------------------------------------
  Random Forest          CLF    rf_classifier.pkl            ✅
  XGBoost Clf            CLF    xgb_classifier.pkl           ✅
  LightGBM Clf           CLF    lgbm_classifier.pkl          ✅
  XGBoost Regressor      RUL    xgb_regressor.pkl            ✅
  LSTM V2                RUL    lstm_rul_best_v2.keras       ✅

-----------------------------------------------------------------
  DATA SHAPES
------------------------

Classifier Comprehensive Evaluation

RUL Comprehensive Evaluation